<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 4 (a) — Chat With Your Own PDF

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Build

A chatbot that answers questions about **a PDF you choose** — grounded in the document, citing the
page it read, and honest enough to say "I don't know".

```
your PDF  →  pages  →  chunks (+ page number)  →  embed  →  store        (once)
your question  →  retrieve  →  grounded prompt  →  answer + page numbers  (every message)
```

1. Get a PDF into Colab and pull the text out of it
2. Chunk it so every chunk remembers **which page** it came from
3. Index the chunks and write `retrieve()`
4. Build a grounded LCEL chain that **refuses** when the answer isn't there
5. Print **page citations** under every answer
6. Wrap the whole thing in a **`gr.ChatInterface`** with a public link
7. *(stretch)* handle follow-up questions, and skip the model call when nothing is relevant

> **You need an OpenAI key** from step 4 onwards. Steps 1–3 run without one — the embedding model
> is local, exactly as in class.

---

## 1. Setup

Run these three cells. Nothing to write yet.

In [ ]:
# PROVIDED - just run this cell.
!pip install -q langchain langchain-openai chromadb sentence-transformers langchain-text-splitters pypdf gradio

In [ ]:
# PROVIDED - just run this cell.
import os
from getpass import getpass

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
import chromadb
import gradio as gr

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chroma = chromadb.Client()
print("Ready")

In [ ]:
# PROVIDED - only needed from step 4 onwards. Press Enter to skip for now.
key = getpass("OpenAI API Key (Enter to skip): ")
if key:
    os.environ["OPENAI_API_KEY"] = key
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    print("Key loaded")
else:
    print("No key - steps 1 to 3 will still work")

---

## 2. Get a PDF into Colab

Two ways. **Run one of them.**

In [ ]:
# PROVIDED - Option A: upload a PDF from your own computer.
# A "Choose Files" button appears - pick a PDF and wait for the upload to finish.
from google.colab import files

uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]

print("Using:", PDF_PATH)

In [ ]:
# PROVIDED - Option B: no PDF handy? Run this instead of the cell above.
# Downloads the paper that introduced RAG - so you can ask a RAG bot about RAG.
import requests

PDF_PATH = "rag_paper.pdf"
url = "https://arxiv.org/pdf/2005.11401"

with open(PDF_PATH, "wb") as f:
    f.write(requests.get(url, timeout=60).content)

print("Downloaded:", PDF_PATH)

> ⚠️ **Pick a PDF with real text in it.** If your PDF is a **scan** (a photo of pages), there is no
> text layer to extract and every page will come out empty — you'd need OCR, which is a different
> lesson. Step 1 makes you check this before you build anything on top of it.
>
> A 10–30 page document works best. A 400-page book will take a while to embed.

---

## 3. The tools you have

| What | How |
|---|---|
| read a PDF | `PdfReader(path).pages` → `page.extract_text()` |
| split text | `RecursiveCharacterTextSplitter(chunk_size=…, chunk_overlap=…)` |
| split **with metadata** | `.create_documents(list_of_texts, metadatas=[{...}, ...])` → `Document(page_content, metadata)` |
| make vectors | `embedder.encode(list_of_texts).tolist()` — the `.tolist()` matters, Chroma won't take numpy |
| store | `collection.add(ids=…, embeddings=…, documents=…, metadatas=…)` |
| search | `collection.query(query_embeddings=…, n_results=…)` → `["documents"][0]`, `["metadatas"][0]`, `["distances"][0]` |
| your function as a chain link | `RunnableLambda(your_function)` |
| a chain | `prompt \| llm \| StrOutputParser()` |
| a chat UI | `gr.ChatInterface(fn).launch(share=True)` — `fn(message, history)` |

Everything from here is yours to write.

In [ ]:
# Step 1 - Pull the text out of the PDF.
# Read PDF_PATH, get one string per page, and print:
#   - how many pages you got
#   - the first 300 characters of page 1
# LOOK at what you printed. If it is empty or gibberish, your PDF is scanned - get a different one
# before going any further.

In [ ]:
# Step 2 - Chunk the pages, keeping the page number, then index them.
# Each chunk must carry the page it came from - that is what you will cite later.
# Print how many chunks you got, and the metadata of one of them to prove the page number is there.

In [ ]:
# Step 3 - Write retrieve(question, n_results=4) that returns the most relevant chunk texts.
# Test it with a question about your PDF, and print what comes back.
# Does it look like the right part of the document?

In [ ]:
# Step 4 - Build a grounded chain.
# Write a prompt with two rules: use ONLY the context, and if the answer is not there reply
# exactly "I don't know based on this document."
# Wrap retrieve in RunnableLambda so it becomes part of the chain.
# Then ask it something your PDF answers, and something it cannot possibly know.

In [ ]:
# Step 5 - Cite the pages.
# Write a function that returns BOTH the answer and the page numbers it came from.
# The page numbers must come from the metadata you stored - never from the model.
# Print an answer with its pages underneath, then open the PDF at that page and check it.

In [ ]:
# Step 6 - Wrap it in a chat interface.  <-- the main event
# def chat(message, history): ... return the answer with its page citations
# gr.ChatInterface(chat, title=..., examples=[...]).launch(share=True)
# Give it 3-4 examples, and make one of them a question your PDF CANNOT answer.
# Open the public link on your phone.

In [ ]:
# Step 7 (stretch) - Handle follow-up questions.
# Ask something, then ask a follow-up that only makes sense in context ("and what about the second one?").
# Watch it retrieve badly. Then fix it: use the history to rewrite the follow-up into a standalone
# search query BEFORE retrieving - but still answer the question the user actually typed.

In [ ]:
# Step 8 (stretch) - Don't pay for hopeless questions.
# Chroma gives you distances, and lower means closer. Print the best distance for a question your
# PDF answers, and for one it does not. Then return "I don't know" WITHOUT calling the model
# when the best chunk is further away than a cut-off you choose from those two numbers.

---

**When it misbehaves:**

| What you see | What it means |
|---|---|
| `extract_text()` returns `''` for every page | your PDF is **scanned** — it's a picture of text, with no text layer. Pick a different PDF (or you'd need OCR) |
| `ValueError` about embeddings on `add()` | you passed a numpy array — add `.tolist()` |
| `IDs already exist` | you ran `add()` twice; use a new collection name or restart the runtime |
| `Expected metadata value to be str, int, float or bool` | Chroma metadata can't hold lists or `None` — page numbers are fine, keep it simple |
| `results["documents"]` looks doubly nested | it is — `query` takes a *batch*, so index `[0]` |
| The bot answers from general knowledge, not your PDF | your context probably isn't reaching the prompt — print it and check |
| It answers a question your PDF can't possibly cover | the grounding rules aren't strict enough, or `n_results` is pulling in junk |
| Answers are right but citations look wrong | you're citing a different query than you answered — check you use **one** `query()` result for both |
| `AuthenticationError` | re-run the key cell |

---

### ✅ What you practised

| Idea | The one-liner |
|---|---|
| **Real documents are messy** | always print the extracted text before you trust it — a scanned PDF gives you nothing |
| **Metadata at split time** | the page number has to be attached before embedding, or it's gone for good |
| **`RunnableLambda`** | your own `retrieve()` is a LangChain component the moment you wrap it |
| **Grounding** | "use ONLY this context" + a fixed refusal is what separates reading from guessing |
| **Citations are retrieved, not generated** | the page number comes from your metadata; never ask the model for it |
| **Follow-ups don't retrieve** | "what about the second one?" has nothing searchable in it until you rewrite it |

**Finished early?**
1. **Two PDFs, one index.** Add a `source` field alongside `page`, index a second PDF, and cite both file and page.
2. **Measure it.** Write 10 questions you know the answers to, and count how often the right page lands in the top-k. Change `chunk_size` and see if the number moves.
3. **Show your work.** Print the retrieved chunks in the chat window under a "Sources" heading, so you can see what the answer was built from.
4. **Give it away.** `share=True` prints a public link that lives for 72 hours — send it to someone and watch what they ask.